# Flow Rate and Performance Comparison

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
df = pd.read_csv('flow_rate_comparison.csv')
df.head()

,sys_id,node_id_g2,node_name,comm_size_g2,elapsed_time_g2,flow_rate_g2,remaining_comms_g2,node_id_ns3,comm_size_ns3,elapsed_time_ns3,flow_rate_ns3,remaining_comms_ns3
0,0,8,mb0.transformer.0.mha.qkv@0_X1COMM,2.684355e+08,210798402,1.273423,10,8,2.684355e+08,218877906,1.226416,10
1,0,28,mb0.transformer.0.mha.dwqkv@0_X2_COMM,2.684355e+08,495855345,0.541358,0,28,2.684355e+08,289460543,0.927365,0
2,0,16,mb0.transformer.0.mha.o@0_X1COMM,4.294967e+09,402578417,10.668648,9,16,4.294967e+09,239952107,17.899269,9
3,0,56,mb0.transformer.0.ffn.x00@0_X1COMM,2.684355e+08,435541404,0.616326,7,56,2.684355e+08,277068868,0.968840,7
4,0,67,mb0.transformer.0.ffn.x01@0_X1COMM,2.684355e+08,405952712,0.661248,0,67,2.684355e+08,339826379,0.789919,0


## Elapsed Time Comparison (g2 vs ns3)

In [3]:
fig = px.scatter(df, x='elapsed_time_g2', y='elapsed_time_ns3', 
                 hover_data=['node_name', 'sys_id'],
                 title='Elapsed Time Comparison: g2 vs ns3')
fig.add_shape(type='line', x0=df['elapsed_time_g2'].min(), y0=df['elapsed_time_g2'].min(), x1=df['elapsed_time_g2'].max(), y1=df['elapsed_time_g2'].max(), line=dict(color='red', dash='dash'))
fig.show()

## Flow Rate Comparison (g2 vs ns3)

In [4]:
fig = px.scatter(df, x='flow_rate_g2', y='flow_rate_ns3', 
                 hover_data=['node_name', 'sys_id'],
                 title='Flow Rate Comparison: g2 vs ns3')
fig.add_shape(type='line', x0=df['flow_rate_g2'].min(), y0=df['flow_rate_g2'].min(), x1=df['flow_rate_g2'].max(), y1=df['flow_rate_g2'].max(), line=dict(color='red', dash='dash'))
fig.show()

## Remaining Communications Comparison (g2 vs ns3)

In [5]:
fig = px.scatter(df, x='remaining_comms_g2', y='remaining_comms_ns3', 
                 hover_data=['node_name', 'sys_id'],
                 title='Remaining Communications Comparison: g2 vs ns3')
fig.add_shape(type='line', x0=df['remaining_comms_g2'].min(), y0=df['remaining_comms_g2'].min(), x1=df['remaining_comms_g2'].max(), y1=df['remaining_comms_g2'].max(), line=dict(color='red', dash='dash'))
fig.show()

This plot shows the difference in elapsed time between `g2` and `ns3` against the average number of remaining communications for each node.

-   **Positive y-values** indicate that `ns3` was faster than `g2` for that particular node.
-   **Negative y-values** indicate that `g2` was faster.
-   The x-axis represents the average number of communication tasks that were pending after a given node's execution.

By observing the trend, we can see if there is a correlation between the number of remaining communications and the performance difference.

In [6]:
## Performance vs. Remaining Communications
# Calculate the difference in elapsed time and the average number of remaining communications
df['elapsed_time_diff'] = df['elapsed_time_g2'] - df['elapsed_time_ns3']
df['avg_remaining_comms'] = (df['remaining_comms_g2'] + df['remaining_comms_ns3']) / 2

# Create a scatter plot to show the correlation
fig = px.scatter(df, 
                 x='avg_remaining_comms', 
                 y='elapsed_time_diff', 
                 hover_data=['node_name', 'sys_id'],
                 title='Elapsed Time Difference vs. Average Remaining Communications',
                 labels={'avg_remaining_comms': 'Average Remaining Communications', 'elapsed_time_diff': 'Elapsed Time Difference (g2 - ns3)'})

# Add a horizontal line at y=0 to show where performance is equal
fig.add_hline(y=0, line=dict(color='red', dash='dash'))

fig.show()

# Another way to visualize this is to color the points based on which simulator was faster.
df['winner'] = df['elapsed_time_diff'].apply(lambda x: 'ns3_faster' if x > 0 else ('g2_faster' if x < 0 else 'equal'))

fig = px.scatter(df, 
                 x='avg_remaining_comms', 
                 y='elapsed_time_diff', 
                 color='winner',
                 hover_data=['node_name', 'sys_id'],
                 title='Performance Difference vs. Remaining Communications',
                 labels={'avg_remaining_comms': 'Average Remaining Communications', 'elapsed_time_diff': 'Elapsed Time Difference (g2 - ns3)'})

fig.add_hline(y=0, line=dict(color='black', dash='dash'))

fig.show()

In [12]:
len(df[df['flow_rate_diff']>0])/(len(df[df['flow_rate_diff']<0])+ len(df[df['flow_rate_diff']>0]))

0.125

In [11]:
len(df[df['flow_rate_diff']<0])

168

In [7]:
## Flow Rate vs. Remaining Communications
# Calculate the difference in flow rate
df['flow_rate_diff'] = df['flow_rate_g2'] - df['flow_rate_ns3']

# Create a scatter plot to show the correlation
fig = px.scatter(df, 
                 x='avg_remaining_comms', 
                 y='flow_rate_diff', 
                 hover_data=['node_name', 'sys_id'],
                 title='Flow Rate Difference vs. Average Remaining Communications',
                 labels={'avg_remaining_comms': 'Average Remaining Communications', 'flow_rate_diff': 'Flow Rate Difference (g2 - ns3)'})

# Add a horizontal line at y=0 to show where flow rate is equal
fig.add_hline(y=0, line=dict(color='red', dash='dash'))

fig.show()

# Color the points based on which simulator had a higher flow rate.
df['flow_rate_winner'] = df['flow_rate_diff'].apply(lambda x: 'g2_higher_flow' if x > 0 else ('ns3_higher_flow' if x < 0 else 'equal'))

fig = px.scatter(df, 
                 x='avg_remaining_comms', 
                 y='flow_rate_diff', 
                 color='flow_rate_winner',
                 hover_data=['node_name', 'sys_id'],
                 title='Flow Rate Difference vs. Remaining Communications',
                 labels={'avg_remaining_comms': 'Average Remaining Communications', 'flow_rate_diff': 'Flow Rate Difference (g2 - ns3)'})

fig.add_hline(y=0, line=dict(color='black', dash='dash'))

fig.show()

This plot shows the difference in flow rate between `g2` and `ns3` against the average number of remaining communications for each node.

-   **Positive y-values** indicate that `g2` had a higher flow rate than `ns3`.
-   **Negative y-values** indicate that `ns3` had a higher flow rate.